In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="lniyaAIMaVl7Cx3O8ICi")
project = rf.workspace("rakib151p").project("vegmining-i6j82")
version = project.version(6)
dataset = version.download("yolov11")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 38.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 63.1 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigque


Extracting Dataset Version Zip to VegMining-6 in yolov11:: 100%|██████████| 9584/9584 [00:01<00:00, 9533.44it/s] 


In [5]:
!pip install -q roboflow ultralytics scikit-learn pyyaml

import os
import shutil
import yaml
import numpy as np
from sklearn.model_selection import KFold
from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.5 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
from sklearn.model_selection import StratifiedKFold

In [10]:
dataset_path = dataset.location 

images_path = os.path.join(dataset_path, "train", "images")
labels_path = os.path.join(dataset_path, "train", "labels")

CLASS_NAMES = [
    'defected_Potato',
    'defected_bitter_gourd',
    'defected_brinjal',
    'defected_capsicum',
    'defected_onion',
    'defected_pointed_gourd',
    'defected_tomato',
    'fresh_bitter_gourd',
    'fresh_brinjal',
    'fresh_capsicum',
    'fresh_onion',
    'fresh_pointed_gourd',
    'fresh_potato',
    'fresh_tomato'
]
NC = len(CLASS_NAMES)

def is_image_file(fname):
    return fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))

def label_name_from_image(img_name):
    return os.path.splitext(img_name)[0] + ".txt"

def safe_mkdir(path):
    os.makedirs(path, exist_ok=True)

def remove_if_exists(path):
    if os.path.exists(path):
        shutil.rmtree(path)

def get_image_class(img_name):
    """
    Reads the YOLO label file and returns the first class id.
    Assumes one main class per image.
    """
    lbl_path = os.path.join(labels_path, label_name_from_image(img_name))
    if not os.path.exists(lbl_path):
        return -1  # or skip these images if preferred

    with open(lbl_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    if not lines:
        return -1

    class_id = int(lines[0].split()[0])
    return class_id

# only actual images
all_images = sorted([f for f in os.listdir(images_path) if is_image_file(f)])

# keep only images with valid labels for stratification
images = []
y = []

for img in all_images:
    cls = get_image_class(img)
    if cls != -1:
        images.append(img)
        y.append(cls)

images = np.array(images)
y = np.array(y)



In [11]:
# Stratified K-Fold setup
k = 10
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

results = []

for i in range(1, k + 1):
    remove_if_exists(f"fold_{i}")

remove_if_exists("cv_runs")

# Cross-validation loop
for fold, (train_idx, val_idx) in enumerate(skf.split(images, y), start=1):
    print(f"\n🚀 Fold {fold}/{k}")

    fold_path = f"fold_{fold}"

    train_img_dir = os.path.join(fold_path, "train", "images")
    train_lbl_dir = os.path.join(fold_path, "train", "labels")
    val_img_dir = os.path.join(fold_path, "val", "images")
    val_lbl_dir = os.path.join(fold_path, "val", "labels")

    safe_mkdir(train_img_dir)
    safe_mkdir(train_lbl_dir)
    safe_mkdir(val_img_dir)
    safe_mkdir(val_lbl_dir)

    # Train split
    for idx in train_idx:
        img = images[idx]
        src_img = os.path.join(images_path, img)
        lbl = label_name_from_image(img)
        src_lbl = os.path.join(labels_path, lbl)

        shutil.copy2(src_img, os.path.join(train_img_dir, img))
        shutil.copy2(src_lbl, os.path.join(train_lbl_dir, lbl))

    # Validation split
    for idx in val_idx:
        img = images[idx]
        src_img = os.path.join(images_path, img)
        lbl = label_name_from_image(img)
        src_lbl = os.path.join(labels_path, lbl)

        shutil.copy2(src_img, os.path.join(val_img_dir, img))
        shutil.copy2(src_lbl, os.path.join(val_lbl_dir, lbl))

    data_yaml = {
        "path": os.path.abspath(fold_path),
        "train": "train/images",
        "val": "val/images",
        "nc": NC,
        "names": CLASS_NAMES
    }

    yaml_file = os.path.join(fold_path, "data.yaml")
    with open(yaml_file, "w") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False)

    model = YOLO("yolo11n.pt")

    model.train(
        data=yaml_file,
        epochs=50,
        imgsz=640,
        project="cv_runs",
        name=f"fold_{fold}",
        exist_ok=True,
        verbose=False
    )

    val_metrics = model.val(
        data=yaml_file,
        split="val",
        imgsz=640,
        project="cv_runs",
        name=f"fold_{fold}_val",
        exist_ok=True,
        verbose=False
    )

    results.append({
        "fold": fold,
        "precision": float(val_metrics.box.mp),
        "recall": float(val_metrics.box.mr),
        "mAP50": float(val_metrics.box.map50),
        "mAP50_95": float(val_metrics.box.map)
    })

print("\n📊 Fold-wise Results:")
for r in results:
    print(
        f"Fold {r['fold']}: "
        f"Precision={r['precision']:.4f}, "
        f"Recall={r['recall']:.4f}, "
        f"mAP50={r['mAP50']:.4f}, "
        f"mAP50-95={r['mAP50_95']:.4f}"
    )

avg = {
    "precision": np.mean([r["precision"] for r in results]),
    "recall": np.mean([r["recall"] for r in results]),
    "mAP50": np.mean([r["mAP50"] for r in results]),
    "mAP50-95": np.mean([r["mAP50_95"] for r in results]),
}

std = {
    "precision": np.std([r["precision"] for r in results]),
    "recall": np.std([r["recall"] for r in results]),
    "mAP50": np.std([r["mAP50"] for r in results]),
    "mAP50-95": np.std([r["mAP50_95"] for r in results]),
}

print("\n📊 Final 10-Fold Results:")
for key in avg:
    print(f"{key}: {avg[key]:.4f} ± {std[key]:.4f}")


🚀 Fold 1/10
Ultralytics 8.4.32 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=fold_1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fold_1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective